<br>

# 基于 PyTorch 实现残差神经网络 ResNet 

<br>

## 0. 概述

<br>

<font color=black size=3 face=雅黑>　　在本节实验中，我们将基于 PyTorch 实现残差神经网络 ResNet，并在图片数据集（CIFAR-10）上进行训练和测试。
    
<br>
    
<font color=black size=3 face=雅黑>　　具体包括如下几个部分：
     
<font color=black size=3 face=雅黑>　　　(1) 学习残差神经网络，特别是 Block 的概念；

<font color=black size=3 face=雅黑>　　　(2) 构建残差神经网络，并基于此实现 CIFAR-10 的训练与测试。

<br>
    
<font color=black size=3 face=雅黑>　　　**请大家注意，由于本次实验的模型相对比较复杂，对云服务器资源要求较高，如发生资源不够或速度较慢的问题，同学们可以在课上两人一组，共用一个登录账号展开实验。**

<br>
    
<font color=black size=2 face=雅黑>　　Ref： https://arxiv.org/pdf/1512.03385.pdf
    
<font color=black size=2 face=雅黑>　　　　　https://zhuanlan.zhihu.com/p/106764370

<br>

## 1. 数据集介绍

<br>
    
<font color=black size=3 face=雅黑>　　官方说明及下载地址：http://www.cs.toronto.edu/~kriz/cifar.html

<br>

### 1.1 数据集准备

<br>

<font color=black size=3 face=雅黑>　　我们首先来准备数据集，方法与之前类似，原始数据可直接从网上下载。
   
<br>

In [2]:
import torch
import torchvision
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

torch.manual_seed(1)

In [3]:
batch_size = 250  # 设置训练集和测试集的 batch size，即每批次将参与运算的样本数

# 训练集
train_set = torchvision.datasets.CIFAR10('./dataset_cifar10', train=True, download=True,
                                       transform=torchvision.transforms.Compose([
                                           torchvision.transforms.ToTensor(),
                                           torchvision.transforms.Normalize(
                                               (0.4914,0.4822,0.4465), (0.2023,0.1994,0.2010)
                                           )
                                       ])
)

# 测试集
test_set = torchvision.datasets.CIFAR10('./dataset_cifar10', train=False, download=True,
                                      transform=torchvision.transforms.Compose([
                                          torchvision.transforms.ToTensor(),
                                          torchvision.transforms.Normalize(
                                              (0.4914,0.4822,0.4465), (0.2023,0.1994,0.2010)
                                          )
                                      ]))

train_loader = torch.utils.data.DataLoader(train_set, batch_size=batch_size, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_set, batch_size=batch_size, shuffle=True)

<br>

### 1.2 简单分类：CIFAR-10

<br>

<font color=black size=3 face=雅黑>　　下面我们首用实验二中定义过的卷积神经网络在 CIFAR-10 数据集上训练并测试，方便后续与残差神经网络对比。
   
<br>

In [4]:
class CNN5(nn.Module):
    def __init__(self):
        super(CNN5, self).__init__()
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=6, kernel_size=5)  # in_channels 由 1 改变为 3
        self.conv2 = nn.Conv2d(in_channels=6, out_channels=12, kernel_size=5)
        
        self.fc1 = nn.Linear(in_features=12*5*5, out_features=120)  # in_features 由 12*4*4 改变为 12*5*5
        self.fc2 = nn.Linear(in_features=120, out_features=60)
        self.out = nn.Linear(in_features=60, out_features=10)
        
        
    def forward(self, t):
        
        # conv1
        t = self.conv1(t)
        t = F.relu(t)  
        t = F.max_pool2d(t, kernel_size=2, stride=2)  
        
        # conv2
        t = self.conv2(t)
        t = F.relu(t)
        t = F.max_pool2d(t, kernel_size=2, stride=2)      
       
        t = t.reshape(batch_size, 12*5*5)  # dim1 由 12*4*4 改变为 12*5*5

        # fc1
        t = self.fc1(t)
        t = F.relu(t)
        
        # fc2
        t = self.fc2(t)
        t = F.relu(t)
        
        # output layer
        t = self.out(t)
        
        return t

In [5]:
network = CNN5()
network.cuda()

loss_func = nn.CrossEntropyLoss()  # 损失函数
optimizer = optim.SGD(network.parameters(), lr=0.1)  # 优化器

def get_num_correct(preds, labels):  # get the number of correct times
    return preds.argmax(dim=1).eq(labels).sum().item()

<br>

<font color=black size=3 face=雅黑>　　开始训练
   
<br>

In [6]:
total_epochs = 4

for epoch in range(total_epochs):

    total_loss = 0
    total_train_correct = 0

    for batch in train_loader:         
        images, labels = batch
        images = images.cuda()
        labels = labels.cuda()
        preds = network(images)
        loss = loss_func(preds, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()       
  
        total_loss += loss.item()
        total_train_correct += get_num_correct(preds, labels)
    
    print("epoch:", epoch, 
          "correct times:", total_train_correct,
          f"training accuracy:", "%.3f" %(total_train_correct/len(train_set)*100), "%", 
          "total_loss:", "%.3f" %(total_loss/len(train_set)*batch_size))

epoch: 0 correct times: 11743 training accuracy: 23.486 % total_loss: 2.070
epoch: 1 correct times: 19430 training accuracy: 38.860 % total_loss: 1.694
epoch: 2 correct times: 22621 training accuracy: 45.242 % total_loss: 1.524
epoch: 3 correct times: 24393 training accuracy: 48.786 % total_loss: 1.430


<br>

<font color=black size=3 face=雅黑>　　测试结果（如进一步增加训练周期，准确率还会进一步上升）
   
<br>

In [8]:
total_test_correct = 0
total_loss = 0

for batch in test_loader:
    images, labels = batch
    images = images.cuda()
    labels = labels.cuda()
    preds = network(images)
    loss = loss_func(preds, labels)

    total_loss += loss
    total_test_correct += get_num_correct(preds, labels)
    
print("correct times:", total_test_correct, 
      f"test accuracy:", "%.3f" %(total_test_correct/len(test_set)*100), "%",
      "total_loss:", "%.3f" %(total_loss/len(test_set)*batch_size))

correct times: 4767 test accuracy: 47.670 % total_loss: 1.487


<br>

## 2. 残差神经网络

<br>

### 2.1 残差神经网络基础

<br>

<font color=black size=3 face=雅黑>

<br>
<font color=black size=3 face=雅黑>　　理论上，增加神经网络准确率的一个有效方法即增加神经网络的深度（层数），例如从上面的 6 层神经网络增加至 20 层左右。网络的深度越深，可抽取的特征层次就越丰富越抽象。   

<br>
    
<font color=black size=3 face=雅黑>　　然而，事实证明有时网络层数并不是越深越好。如下图所示，是两个普通的深层卷积神经网络 (plain CNN) 在 CIFAR-10 上的训练和测试结果。两个神经网络的深度分别是 20 层和 56 层。

<font color=black size=3 face=雅黑>　　和正常结构相比，残差结构 (residual block) 多了右侧的曲线，我们将这个曲线称作 shortcut connection。它将上一层（或几层）的输出“跳接”到本层，在本层的计算结果进入到激活函数 ReLU 之前与之相加，并将相加的结果一起输入到激活函数作为本层的最终输出。深度残差网络正是由许多这样的残差结构构成的。
    
<br>
    
<font color=black size=3 face=雅黑>　　下图展示了一个完整的 ResNet（最右侧的网络，共33个卷积层，1个全连接层）。为了方便对比，图中也画出了 VGG-19 和不带 shortcut connections 的 plain CNN。
    

<br>
    
<font color=black size=3 face=雅黑>　　在《Deep Residual Learning for Image Recognition》一文中，一共提出了五种 ResNet 网络结构，分别是 18 层、34 层、50层、101层和 152 层。五种结构的细节如下所示：
    
<br>

<br>
    
<font color=black size=3 face=雅黑>　　下面我们就来构建这五种 ResNet。虽然它们深度不同，但都有一个共同特点，即都是由两种简单的残差结构 (residual block) 组成的。
    
<br>
    
<font color=black size=3 face=雅黑>　　(1) 第一种残差结构（适用于 ResNet18 和 ResNet34）：含两层 kernel_size=3 的卷积层，block 内输出通道数维持不变；
    
<font color=black size=3 face=雅黑>　　(2) 第二种残差结构（适用于 ResNet50、ResNet101 和 ResNet152）：含三层卷积层，kernel_size 分别取 1、3、1，最后一层
    
<font color=black size=3 face=雅黑>　　　　输出通道数扩展为原先的 4 倍。
    
    
<br>
    
    

<br>

### 2.2 构建两种 Residual Blocks

<br>
    
<font color=black size=3 face=雅黑>　　(1) 第一种残差结构：BasicBlock；
    
<font color=black size=3 face=雅黑>　　(2) 第二种残差结构：BottleneckBlock。
    

<br>

In [9]:
class BasicBlock(nn.Module):
    channel_expansion = 1  # {扩展后的最终输出通道数} / {扩展前的输出通道数（blk_mid_channels）}
    
    def __init__(self, blk_in_channels, blk_mid_channels, stride=1):
        super(BasicBlock, self).__init__()
        
        self.conv1 = nn.Conv2d(in_channels=blk_in_channels,
                               out_channels=blk_mid_channels,  
                               kernel_size=3,
                               padding=1,
                               stride=stride)
        self.bn1 = nn.BatchNorm2d(blk_mid_channels)
        
        self.conv2 = nn.Conv2d(in_channels=blk_mid_channels,
                               out_channels=blk_mid_channels*self.channel_expansion,
                               kernel_size=3, 
                               padding=1, 
                               stride=1)
        self.bn2 = nn.BatchNorm2d(blk_mid_channels*self.channel_expansion)
        
        # 实现 shortcut connection：        
        if stride != 1 or blk_in_channels != self.channel_expansion*blk_mid_channels: # 形状不同
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels=blk_in_channels,
                          out_channels=self.channel_expansion*blk_mid_channels,
                          kernel_size=1,
                          padding=0,
                          stride=stride),
                nn.BatchNorm2d(self.channel_expansion*blk_mid_channels)
            )
        else:
            self.shortcut = nn.Sequential()
            
        
    def forward(self, t):
        ################### Please finish the following code ###################
        out = self.conv1(t)
        out = self.bn1(out)
        out = F.relu(out)
        
        out = self.conv2(out)
        out = self.bn2(out)
        
        out += self.shortcut(t)
        out = F.relu(out)
        ########################################################################
        
        return out

In [10]:
class BottleneckBlock(nn.Module):
    channel_expansion = 4  # {扩展后的最终输出通道数} / {扩展前的输出通道数（blk_mid_channels）}
    
    def __init__(self, blk_in_channels, blk_mid_channels, stride=1):
        super(BottleneckBlock, self).__init__()
        
        self.conv1 = nn.Conv2d(in_channels=blk_in_channels,
                               out_channels=blk_mid_channels,
                               kernel_size=1,
                               padding=0,
                               stride=1)
        self.bn1 = nn.BatchNorm2d(blk_mid_channels)
        
        self.conv2 = nn.Conv2d(in_channels=blk_mid_channels,
                               out_channels=blk_mid_channels,
                               kernel_size=3,
                               padding=1,
                               stride=stride)
        self.bn2 = nn.BatchNorm2d(blk_mid_channels)
        
        self.conv3 = nn.Conv2d(in_channels=blk_mid_channels,
                               out_channels=blk_mid_channels*self.channel_expansion, 
                               kernel_size=1,
                               padding=0,
                               stride=1)
        self.bn3 = nn.BatchNorm2d(blk_mid_channels*self.channel_expansion)
        
        # 实现 shortcut connection：
        if stride != 1 or blk_in_channels != blk_mid_channels*self.channel_expansion: # 形状不同
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels=blk_in_channels,
                          out_channels=blk_mid_channels*self.channel_expansion,
                          kernel_size=1,
                          padding=0,
                          stride=stride), 
                nn.BatchNorm2d(blk_mid_channels*self.channel_expansion)
            )
        else: 
            self.shortcut = nn.Sequential()
            
        
    def forward(self, t):
        
        ################### Please finish the following code ###################
        out = self.conv1(t)
        out = self.bn1(out)
        out = F.relu(out)
        
        out = self.conv2(out)
        out = self.bn2(out)
        out = F.relu(out)
        
        out = self.conv3(out)
        out = self.bn3(out)
        
        out += self.shortcut(t)
        out = F.relu(out)
         
        ########################################################################
        
        return out

<br>

### 2.3 构建完整的残差神经网络

<br>
      
<font color=black size=3 face=雅黑>　　接下来我们来基于 BasicBlock 和 BottleneckBlock 构建完整的残差神经网络。
    

<br>

<font color=black size=3 face=雅黑>　　稍后我们将以 ResNet18 为例在 CIFAR-10 上训练并测试：
    
<font color=black size=3 face=雅黑>　　（请注意：该训练对服务器资源要求较高，因为实验课上很多同学同时使用，有时服务器会存在支持不了的情况。如果出现该情况，请同学们不要着急，先把代码填空完成，稍后再尝试运行。）


In [11]:
class ResNet(nn.Module):
    def __init__(self, block, num_blocks, num_classes):
        super(ResNet, self).__init__()
        
        self.residual_layers = 4  # 每个 "residual layer" 含多个 blocks，对应上面列表中的一行 (即 conv2_x, conv3_x, conv4_x 或 conv5_x)
        self.blk1_in_channels = 32  # 按照上面的列表，此处应填 64，但由于大网络训练起来耗时太长，此处我们酌情把全部通道都除以 2
        self.blk_mid_channels = [32, 64, 128, 256]  # 原先的通道数：[64, 128, 256, 512]
        self.blk_channels = [self.blk1_in_channels] + self.blk_mid_channels  # [32, 32, 64, 128, 256]
        self.blk_stride = [1,2,2,2]  # 每个 residual layer 的 stride
        
        self.blk_channel_expansion = block.channel_expansion
        
        # 第一个卷积层（独立于 residual layers 之外）
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=self.blk_channels[0], kernel_size=3, padding=1, stride=1)
        self.bn1 = nn.BatchNorm2d(self.blk1_in_channels) 
        
        # residual layers (打包在 self.layers 中)
        self.layers = nn.Sequential()
        for i in range(self.residual_layers):
            blk_in_channels = self.blk_channels[i] if i==0 else self.blk_channels[i]*block.channel_expansion
            blk_mid_channels = self.blk_channels[i+1]
            self.layers.add_module(f"residule_layer{i}", 
                                   self._make_layer(block=block,  # block 种类：BasicBlock 或 BottleneckBlock
                                                    blk_in_channels=blk_in_channels,
                                                    blk_mid_channels=blk_mid_channels, 
                                                    num_blocks=num_blocks[i],  # 该 residual layer 有多少个 blocks
                                                    stride=self.blk_stride[i])
            )
        
        # 最后的全连接层
        self.linear = nn.Linear(in_features=self.blk_channels[self.residual_layers]*block.channel_expansion, 
                                out_features=num_classes)
        
        
    def _make_layer(self, block, blk_in_channels, blk_mid_channels, num_blocks, stride):
        block_list = []
        stride_list = [stride] + [1]*(num_blocks-1)  # 每个 block 的 stride
        
        for block_idx in range(num_blocks):
            if block_idx != 0:  # 对于 residual layer 中非第一个 block: 调整其 blk_in_channels
                blk_in_channels = blk_mid_channels*block.channel_expansion
            block_list.append(
                block(blk_in_channels=blk_in_channels, 
                      blk_mid_channels=blk_mid_channels, 
                      stride=stride_list[block_idx])
            )
        
        return nn.Sequential(*block_list)  # 返回一个 residual layer
    
    
    def forward(self, t):
        
        ################### Please finish the following code ###################

        # conv1
        out = self.conv1(t)
        out = self.bn1(out)
        out = F.relu(out)
        
        # "residual layers"（打包在 self.layers 中）
        out = self.layers(out)
        
        
        ########################################################################
        
        
        # 实现最后一个全连接层
        out = F.avg_pool2d(out, 4)  # shape of "out" before pooling (ResNet18): (batch_size, 256, 4, 4)
        out = out.reshape(batch_size, self.blk_mid_channels[self.residual_layers-1]*self.blk_channel_expansion)
        out = self.linear(out)         
        
        return out

<br>

<font color=black size=3 face=雅黑>构建五种 ResNet。
    
<br>

In [12]:
def ResNet18():
    return ResNet(BasicBlock, [2,2,2,2], 10)


def ResNet34():
    return ResNet(BasicBlock, [3,4,6,3], 10)


def ResNet50():
    return ResNet(BottleneckBlock, [3,4,6,3], 10)


def ResNet101():
    return ResNet(BottleneckBlock, [3,4,23,3], 10)


def ResNet152():
    return ResNet(BottleneckBlock, [3,8,36,3], 10)

<br>
      
<font color=black size=3 face=雅黑>　　开始训练前，我们先用下面的代码简单测试一下网络结构和输出结果的形状是不是和预想的一样。
    
<br>

In [13]:
def test_output_shape():
    net = ResNet18()
    x = torch.randn(batch_size,3,32,32)  # 模拟输入
    y = net(x)
    print(net)  # 查看网络结构
    print("")
    print(y.shape)  # 查看输出形状

In [14]:
test_output_shape()

ResNet(
  (conv1): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (layers): Sequential(
    (residule_layer0): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (bn1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv2): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (bn2): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (shortcut): Sequential()
      )
      (1): BasicBlock(
        (conv1): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (bn1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv2): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (bn2): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=T

<br>
      
<font color=black size=3 face=雅黑>　　从显示的结果可以看到，整个网络一共有 8 个 shortcut connections，与上面 ResNet18 的结构图对应。其中，第 3 个、第 5 个、第 7 个 shortcut connection 上额外增加了一次对 x 的 conv/bn 变换。
    
<br>

<br>

### 2.4 训练与测试

<br>

<font color=black size=3 face=雅黑>　　将构建好的 ResNet18 在 CIFAR-10 数据集上训练并测试。

<br>

In [15]:
network = ResNet18()
network = network.cuda()  # 将模型转移到 GPU 上

In [16]:
loss_func = nn.CrossEntropyLoss()  # 损失函数：交叉熵损失
optimizer = torch.optim.Adam(network.parameters(), lr=0.001)  # 优化器

def get_num_correct(preds, labels):  # 计算正确分类的次数
    return preds.argmax(dim=1).eq(labels).sum().item()

In [17]:
total_epochs = 3  # 由于训练耗时较长，这次我们只训练 3 个周期来看一下结果

for epoch in range(total_epochs):

    total_loss = 0
    total_train_correct = 0

    for batch in train_loader:  # 抓取一个 batch
        
        # 读取样本数据        
        images, labels = batch
        images = images.cuda()  # 数据转移到 GPU 上
        labels = labels.cuda()  # 标签转移到 GPU 上
        
        # 完成正向传播，计算损失
        preds = network(images)
        loss = loss_func(preds, labels)
        
        # 偏导归零
        optimizer.zero_grad()
        
        # 反向传播 
        loss.backward()
        
        # 更新参数        
        optimizer.step()
          
        total_loss += loss.item()
        total_train_correct += get_num_correct(preds, labels)
    
    print("epoch: ", epoch, 
          "correct times:", total_train_correct,
          "training accuracy:", "%.3f" %(total_train_correct/len(train_set)*100), "%", 
          "total_loss:", "%.3f" %(total_loss/len(train_set)*batch_size))

epoch:  0 correct times: 26561 training accuracy: 53.122 % total_loss: 1.284
epoch:  1 correct times: 35673 training accuracy: 71.346 % total_loss: 0.806
epoch:  1 correct times: 35673 training accuracy: 71.346 % total_loss: 0.806
epoch:  2 correct times: 39576 training accuracy: 79.152 % total_loss: 0.601
epoch:  2 correct times: 39576 training accuracy: 79.152 % total_loss: 0.601


<br>

<font color=black size=3 face=雅黑>保存模型
    
<br>

In [18]:
torch.save(network.cpu(), "resnet18.pt")

<br>

<font color=black size=3 face=雅黑>在测试集上测试模型
    
<br>

In [21]:
network = ResNet18()
network = torch.load("resnet18.pt", weights_only=False)
num_correct = 0

for i, batch in enumerate(test_loader):
    images, labels = batch    
    preds = network(images)
    
    if i == 0:
        print("preds.shape: ", preds.shape)  # 检查 pred 的形状
        pred_labels = torch.max(preds, dim=1)[1].data.numpy().squeeze()  # 得到全部测试样本的分类结果
        print("pred_labels.shape: ", pred_labels.shape)
        print("predicted labels (first 10 samples): ", pred_labels[:10])  # 打印前 10 个样本的分类结果
        print("real labels (first 10 samples): ", labels[:10])  # 打印前 10 个样本的分类结果
    
    num_correct += get_num_correct(preds, labels)

preds.shape:  torch.Size([250, 10])
pred_labels.shape:  (250,)
predicted labels (first 10 samples):  [7 6 5 0 3 7 5 5 2 7]
real labels (first 10 samples):  tensor([9, 4, 3, 0, 3, 4, 5, 4, 2, 9])


In [22]:
test_accuracy = num_correct/10000

print("test accuracy: ", test_accuracy)

test accuracy:  0.7784


<br>

# 3.  实验报告

<br>

<font color=black size=3 face=雅黑>请同学们在实验报告中完成如下内容：
    
<br>
    
<font color=black size=3 face=雅黑>**“4_pytorch_resnet.ipynb”**：

<br>
    
<font color=black size=3 face=雅黑>　　-　2. 残差神经网络
    
<font color=black size=3 face=雅黑>　　　　请同学们认真阅读这部分内容，补全相关代码；
    
<font color=black size=3 face=雅黑>　　　　在 CIFAR-10 上训练并测试 ResNet18，并展示结果；
      
<br> 
<br>